# 18 — Full topology, predictions from the posterior, and closing of the real-PET fit

Decisions after nb17:

| Decision | Where |
|---|---|
| Fit, therapy and cohort generation use the same model: **full topology**. Predictions from the **posterior**, not the MAP. Sampling and empirical Bayes validation redone | §4, §7 |
| Fix the simulated patient with a non-positive rest of body | §1 |
| Hematocrit prior 0.42 ± 0.04. Is the real blood misfit scanner-dependent after the per-scanner PSF (measurement) or uniform (model)? If model, one nuisance on early plasma clearance | §2 |
| Kidneys: do residuals and fitted $[R_K]$ correlate with collecting-system activity inside the grown mask? Report, propose a fix, do not tune | §3 |
| Spleen: heavy-tailed image error (Student-t) | §5 |
| Lesions: low tail of $[R]$ for non-avid lesions; viable fraction against the RC trend; then empirical Bayes, censored likelihood, patient effect | §6, §7, §8, §9 |
| What the real-PET fit is good for, and its limitations. Then plan (b) | §10 |

**Real PETs: training split only, the loader refuses the test split.** What changed in `src/pbpk_fit.py` compared with nb17: `main_options` now has the full topology, per-scanner recovery in `load_geometry`, tight hematocrit, Student-t on the spleen. Rerunning nb13, nb16 or nb17 without their caches would not reproduce them; their numbers are kept as computed.

**About this notebook.** Final fit of the PBPK model on the real 68Ga PETs of the training split; empirical Bayes population.
- **Inputs:** phantoms; caches `pbpk/nb18_*.pkl`.
- **Outputs:** `data/processed/psma/pbpk/nb18_*` (`nb18_eb_real_population.csv`).
- **Run time:** about 8 min with the caches; about 2 h on 5 processes from scratch.
- **Environment:** .venv.

In [ ]:
import os
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
import json
import multiprocessing as mp
import pickle
import sys
import warnings
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(ROOT / "src"))
import pbpk_fit as F

warnings.filterwarnings("ignore", category=UserWarning)
OUT = ROOT / "data/processed/psma/pbpk"
N_PROCESSES = 5
SEED = 20261105
LIMIT = int(os.environ.get("NB18_LIMIT", 0))
TAG = f"nb18{'_smoke' if LIMIT else ''}"
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
INK, MUTED = "#0b0b0b", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.axisbelow": True, "font.size": 9,
                     "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "axes.edgecolor": "#b8b7ae"})
pd.set_option("display.width", 230, "display.max_columns", 40, "display.max_rows", 150)
ORGANS = list(F.ORGANS)
KEYS = ["kidneys", "liver", "spleen", "f_sal"]


def pool():
    return mp.get_context("fork").Pool(N_PROCESSES)


def run(stage, jobs, chunksize=2):
    with pool() as workers:
        output = workers.map(F.safe_call, [(stage, j) for j in jobs], chunksize=chunksize)
    failed = [x for x in output if isinstance(x, dict) and "error" in x]
    print(stage, "ok", len(output) - len(failed), "of", len(output), *([failed[0]["error"][:160]] if failed else []))
    return [x for x in output if not (isinstance(x, dict) and "error" in x)]


def spread(values):
    logs = np.log(np.asarray(values))
    return 1.4826 * np.median(np.abs(logs - np.median(logs)))


def cached(name, build):
    path = OUT / f"{TAG}_{name}.pkl"
    if path.exists():
        return pickle.loads(path.read_bytes())
    value = build()
    path.write_bytes(pickle.dumps(value))
    return value


split = json.loads((ROOT / "data/processed/psma/psma_68ga_split.json").read_text())
TRAIN, TEST = split["train_studies"], set(split["test_studies"])
assert not TEST & set(TRAIN)
if LIMIT:
    must = ["psma_8effd73b76ee3191_2017-01-28", "psma_daf98725797ce51a_2016-12-17"]
    TRAIN = must + [s for s in TRAIN[::max(len(TRAIN) // LIMIT, 1)] if s not in must][:LIMIT - 2]
SHIFTED_ORGANS = dict(f_sal=(F.PRIORS["f_sal"][0] * 0.5, 0.5), kidneys=(F.PRIORS["kidneys"][0] * 1.35, 0.35), liver=(F.PRIORS["liver"][0] * 1.1, 0.5), spleen=(F.PRIORS["spleen"][0] * 2.4, 0.8))
SHIFTED_LESIONS = F.LesionHyper(float(np.log(60.0)), 0.8, 0.6, float(np.log(0.3)), 0.9, pi_low=0.10, mu_low=float(np.log(3.0)))
null_s = cached("null_measures", lambda: run("stage_measure_subject", [(s, SEED, None) for s in TRAIN]))
shifted_s = cached("shifted_measures", lambda: run("stage_measure_subject", [(s, SEED, SHIFTED_ORGANS, SHIFTED_LESIONS) for s in TRAIN]))
print(len(TRAIN), "training studies")

## 1. The patient with a non-positive rest of body

`psma_8effd73b76ee3191_2017-01-28` and `psma_daf98725797ce51a_2016-12-17` failed with a negative vascular volume of the rest of body (-0.05 to -0.14 L). Both have a TLV above 2 L and a liver of 2.5-3.2 L, but those volumes are not the cause: at any hematocrit up to 0.55 their rest of body stays positive (above 0.9 L). The failure needs a hematocrit of about 0.9. **The optimiser reached it:** the log-normal prior did not bound it, and the plasma volume goes with $1-h$.

**Fix, physical:** hematocrit bounded to 0.20-0.65 in the fit and in the sampler, and the tight prior below. The model raises the error on purpose for impossible inputs, so the check stays. Below, the count of failures of every stage (`run` prints it), and the two patients.

In [ ]:
for study in ["psma_8effd73b76ee3191_2017-01-28", "psma_daf98725797ce51a_2016-12-17"]:
    record = next((r for r in null_s if r["study_id"] == study), None)
    result = F.safe_call(("stage_fit_full", (record, None, None, 0))) if record else None
    print(study, "fit ok" if result and "error" not in result else result, round(result["row"]["hematocrit"], 3) if result and "row" in result else "")

## 2. Blood: hematocrit prior, per-scanner PSF, measurement or model

Hematocrit prior 0.42 ± 0.04, as decided (it replaces the assumed 0.40 ± 0.10; truth drawn from the same). With it the hematocrit no longer absorbs a mismatch, so the mismatch shows.

**Statistic.** `blood_vs_population`: log of the blood datum over the model with this patient's organ densities and with GFR, hematocrit and amount at the population. It is what the patient's nuisance would have to absorb. **Rule, set before looking:** dependent on scanner after the per-scanner PSF means measurement; uniform across scanners means model, and then one nuisance on early plasma clearance, declared as a PSMA-11 against I&T difference.

Scanner scale from the organs (the nb17 offsets) is removed, since a global calibration of the scanner should move blood and organs together. The FWHM per scanner (6.5, 7.5, 6.5 mm) comes from nb13 and is an assumption: the headers give neither the reconstruction nor the filter.

In [ ]:
def blood_arms():
    out = {}
    for psf in ("nominal", "scanner"):
        rows = run("stage_diagnostic_real", [(s, TEST, dict(psf=psf)) for s in TRAIN])
        out[psf] = rows
    return out


real_arms = cached("real_blood_arms", blood_arms)
real0 = real_arms["scanner"]
real_rows = pd.DataFrame([r["row"] for r in real0])
sim0 = cached("sim_first_pass", lambda: run("stage_fit_full", [(r, None, None, 0) for r in null_s]))
sim_rows = pd.DataFrame([r["row"] for r in sim0])
NB17_OFFSETS = dict(Biograph20_mCT=0.096, Biograph64=-0.047, Discovery690=0.011)
tables = {}
for name, rows in [("real, nominal PSF", pd.DataFrame([r["row"] for r in real_arms["nominal"]])), ("real, per-scanner PSF", real_rows)]:
    d = rows.assign(against=rows.blood_vs_population - rows.scanner.map(NB17_OFFSETS))
    tables[name] = d.groupby("scanner").agg(n=("against", "size"), rc_blood=("rc_blood", "median"), median=("against", "median"), se=("against", "sem"), gfr=("gfr", "median"), hematocrit=("hematocrit", "median"), uptake_min=("uptake_min", "median"), z_blood=("z_blood", "median"))
    groups = [x.against.dropna() for _, x in d.groupby("scanner")]
    print(name, "| Kruskal-Wallis p =", f"{stats.kruskal(*groups).pvalue:.1e}", "| all scanners", round(d.against.median(), 3))
sim_scale = pd.Series(F.SCANNER_EFFECT) - np.mean([F.SCANNER_EFFECT[x] for x in sim_rows.scanner])
sim_against = sim_rows.assign(against=sim_rows.blood_vs_population - sim_rows.scanner.map(sim_scale))
tables["simulated, per-scanner PSF"] = sim_against.groupby("scanner").agg(n=("against", "size"), median=("against", "median"), se=("against", "sem"), gfr=("gfr", "median"), hematocrit=("hematocrit", "median"), z_blood=("z_blood", "median"))
print("simulated | Kruskal-Wallis p =", f"{stats.kruskal(*[x.against.dropna() for _, x in sim_against.groupby('scanner')]).pvalue:.1e}")
blood_table = pd.concat(tables)
blood_table.to_csv(OUT / f"{TAG}_blood_by_scanner.csv")
display(blood_table.round(3))
d = real_rows.assign(against=real_rows.blood_vs_population - real_rows.scanner.map(NB17_OFFSETS))
design = pd.get_dummies(d.scanner, dtype=float).assign(log_uptake=np.log(d.uptake_min / 60), log_tlv=np.log1p(d.tlv_ml), log_weight=np.log(d.weight_kg / 80))
keep = d.against.notna()
coef = np.linalg.lstsq(design[keep].to_numpy(), d.against[keep].to_numpy(), rcond=None)[0]
print("With uptake time, TLV, weight in the regression:")
display(pd.Series(coef, design.columns).round(3).to_frame("coefficient").T)
meta = {s: json.loads((F.PHANTOMS / s / "meta.json").read_text())["grid"]["spacing_mm"] for s in d.study_id}
d = d.assign(spacing=[f"{meta[s][0]:.1f}x{meta[s][2]:.1f}" for s in d.study_id], bg_over_raw=d.blood_background / d.blood, raw_rc=np.log(d.blood / d.rc_blood) - np.log(d.blood_corrected))
display(d.groupby(["scanner", "spacing"]).agg(n=("against", "size"), against=("against", "median"), ring_over_raw=("bg_over_raw", "median"), rest_over_ring=("rest", "median")).round(4))
print("Ring over raw blood, by scanner:", d.groupby("scanner").bg_over_raw.median().round(3).to_dict())
fig, ax = plt.subplots(figsize=(5.2, 2.8), constrained_layout=True)
scanners = sorted(d.scanner.unique())
for x, s in enumerate(scanners):
    v = d[d.scanner == s].against.dropna()
    ax.scatter(np.full(len(v), x) + np.random.default_rng(1).uniform(-0.15, 0.15, len(v)), v, s=9, color=COLORS[0], alpha=0.45, linewidths=0)
    ax.plot([x - 0.25, x + 0.25], [v.median()] * 2, color=INK, lw=1.6)
ax.axhline(0, color=MUTED, lw=0.8)
ax.set_xticks(range(len(scanners)), scanners)
ax.set(ylabel="Blood datum over the model at\nthe population nuisance, log")
ax.set_title("The real blood misfit depends on the scanner", loc="left")
fig.savefig(OUT / f"{TAG}_blood_by_scanner.png", dpi=160)

**What the blood datum does from here.** Scanner-dependent means measurement, so no plasma nuisance. A blood offset per scanner enters the population model next to the scanner scale, estimated with the same rule on simulated and real cohorts: the median of the statistic per scanner, accumulated over passes. The consequence is declared: **the blood pool then carries only within-scanner information on GFR, and none on its population level.** Which scanner is right cannot be said from these data.

## 3. Kidneys: the collecting system inside the grown mask

The PBPK kidney has no urine in the collecting system: filtered ligand leaves to the urine state at once. The real datum is the activity of the grown mask, so **any urine in pelvis and proximal ureter is read as kidney uptake and pushes $[R_K]$ up.**

**Where it is.** Label 2 of the phantoms (fluid inside the CT kidney) is close to empty, as printed below: so the pelvis evidently lies outside the CT kidney mask, in the 10 mm shell, inside the concavity (seen on our masks, not checked against the TotalSegmentator documentation). **Measure:** `renal_sinus`, the morphological closing of the CT kidney with radius 15 mm (fixed once, about the half-width of the sinus) minus the kidney, inside the grown mask. `sinus_fraction`: its share of the datum. `sinus_over_parenchyma`: its mean concentration over the median of the functional kidney. `excess_fraction`: activity above the parenchymal level in the sinus, and `shell_excess_fraction` in the whole shell, over the datum. **The simulated PET has no urine there**, so its values are what spill-out and shift alone give. Correlations with the kidney residual and fitted $[R_K]$, also partial on kidney volume. **Reported only: the fit is not changed for this.**

In [ ]:
def kidney_metrics(d):
    d = d.copy()
    d["fluid_fraction"] = d.kidney_fluid_pct / d.kidney_grown
    d["sinus_fraction"] = d.kidney_sinus_pct / d.kidney_grown
    d["sinus_over_parenchyma"] = (d.kidney_sinus_pct / d.kidney_sinus_ml) / d.kidney_parenchyma
    d["excess_fraction"] = d.kidney_sinus_excess_pct / d.kidney_grown
    d["shell_excess_fraction"] = d.kidney_shell_excess_pct / d.kidney_grown
    return d


COLUMNS = ["kidney_grown", "kidney_fluid_pct", "kidney_fluid_ml", "kidney_sinus_pct", "kidney_sinus_ml", "kidney_parenchyma", "kidney_sinus_excess_pct", "kidney_shell_excess_pct"]
KID = kidney_metrics(real_rows)
SIMK = kidney_metrics(pd.DataFrame([{k: r["measure"][k] for k in COLUMNS} | dict(study_id=r["study_id"]) for r in null_s]))
METRICS = ["fluid_fraction", "kidney_fluid_ml", "kidney_sinus_ml", "sinus_fraction", "sinus_over_parenchyma", "excess_fraction", "shell_excess_fraction"]
summary = pd.concat({name: d[METRICS].quantile([0.1, 0.5, 0.9]).T for name, d in [("real", KID), ("simulated", SIMK)]}, axis=1)
summary.to_csv(OUT / f"{TAG}_kidney_summary.csv")
display(summary.round(3))


def partial(x, y, z):
    rx, ry, rz = (stats.rankdata(v) for v in (x, y, z))
    ex = rx - np.polyval(np.polyfit(rz, rx, 1), rz)
    ey = ry - np.polyval(np.polyfit(rz, ry, 1), rz)
    return np.corrcoef(ex, ey)[0, 1]


rows = []
good = KID[np.isfinite(KID.sinus_over_parenchyma) & (KID.kidney_sinus_ml > 1)]
for metric in ["sinus_fraction", "sinus_over_parenchyma", "excess_fraction", "shell_excess_fraction"]:
    for target in ["z_kidneys", "misfit_kidneys", "r_kidneys"]:
        y = np.log(good[target]) if target == "r_kidneys" else good[target]
        r, p = stats.spearmanr(good[metric], y)
        rows.append(dict(metric=metric, target=target, spearman=r, p=p, partial_on_volume=partial(good[metric], y, np.log(good.kidneys_ml)), n=len(good)))
kidney_table = pd.DataFrame(rows)
kidney_table.to_csv(OUT / f"{TAG}_kidney_collecting.csv", index=False)
display(kidney_table.round(3))
q = np.ceil(good.sinus_over_parenchyma.rank(method="first") / len(good) * 4).astype(int)
display(good.groupby(q).agg(n=("r_kidneys", "size"), ratio=("sinus_over_parenchyma", "median"), sinus_fraction=("sinus_fraction", "median"), r_kidneys=("r_kidneys", "median"), z=("z_kidneys", "median"), datum=("datum_kidneys", "median")).round(3))
fig, ax = plt.subplots(figsize=(5, 2.9), constrained_layout=True)
ax.scatter(good.sinus_over_parenchyma, good.r_kidneys, s=10, color=COLORS[0], alpha=0.6, linewidths=0)
ax.set(yscale="log", xlabel="Renal sinus concentration over parenchyma", ylabel=r"Fitted $[R_K]$ (nmol/L)")
ax.set_title("Renal sinus activity against kidney density", loc="left")
fig.savefig(OUT / f"{TAG}_kidney_collecting.png", dpi=160)

## 4. Sampling with the full topology

Metropolis over organ densities and nuisance, 1500 steps, on a random 90 patients of the null cohort, priors of the fit. Compared: Laplace SD, importance sampling (`posterior_sample`, 200 proposals from a Student-t around the MAP, the sampler for the whole cohort), MAP against posterior median. TIAC against the truth of the same topology, with lesions from the joint draw. "Predictive" adds the release rates from their prior, which is what the virtual cohort will use.

In [ ]:
rng = np.random.default_rng(18)
subset = rng.choice(len(null_s), min(90, len(null_s)), replace=False)
sampling = cached("sampling", lambda: run("stage_sampling_full", [(null_s[i], 1500 if not LIMIT else 300, 30 if not LIMIT else 8, None) for i in subset], chunksize=1))
sampled_t = pd.DataFrame([r for x in sampling for r in x["therapy"]])
sampled_p = pd.DataFrame([r for x in sampling for r in x["parameters"]])
print(sampled_t.study_id.nunique(), "patients; acceptance", round(sampled_p.acceptance.median(), 2), "| importance ESS median", round(sampled_p.groupby("study_id").ess.first().median(), 1), "min", round(sampled_p.groupby("study_id").ess.first().min(), 1))
summary = sampled_t.groupby("structure")[["at_map", "median_post", "median_importance", "geo_post", "mean_post", "median_pred", "inside_68", "inside_95"]].median(numeric_only=True).loc[ORGANS]
summary[["inside_68", "inside_95"]] = sampled_t.groupby("structure")[["inside_68", "inside_95"]].mean().loc[ORGANS]
summary.to_csv(OUT / f"{TAG}_sampling_tiac.csv")
display(summary.round(3))
sampled_p["error_map"] = sampled_p["map"] - sampled_p.truth
sampled_p["error_post"] = sampled_p.post_median - sampled_p.truth
sampled_p["sd_ratio"] = sampled_p.post_sd / sampled_p.laplace_sd
sampled_p["importance_shift"] = (sampled_p.importance_median - sampled_p.post_median) / sampled_p.post_sd
sampled_p["importance_sd_ratio"] = sampled_p.importance_sd / sampled_p.post_sd
sampled_p["inside"] = (sampled_p.truth >= sampled_p.q025) & (sampled_p.truth <= sampled_p.q975)
table = sampled_p.groupby("parameter", sort=False)[["error_map", "error_post", "sd_ratio", "importance_shift", "importance_sd_ratio", "inside"]].median(numeric_only=True)
table["inside"] = sampled_p.groupby("parameter", sort=False).inside.mean()
table.to_csv(OUT / f"{TAG}_sampling_parameters.csv")
display(table.round(3))
paired = sampled_t.assign(post_over_map=sampled_t.median_post / sampled_t.at_map).groupby("structure").post_over_map.agg(["median", "sem"]).loc[ORGANS]
display(paired.round(3))

## 5. Spleen: Student-t

nb17: the spleen error has a tail no Gaussian term fits (kurtosis about 10). A Student-t fitted to the image error of nb17 plus the noise (SD 0.10) gives $\nu=2.85$, scale 0.119, so $\nu=3$, scale 0.12 (`SPLEEN_T`). **Measured on the simulated image error, not tuned on coverage**; the shift behind it (SD 4 mm) is the assumption of nb13. The Laplace SD of a Student-t likelihood is not its width, so the check is on the importance sample (§7) and here on $z$. Arm with the Gaussian term as in nb17, same patients.

In [ ]:
gauss = cached("gaussian_spleen", lambda: run("stage_fit_full", [(r, dict(robust=None, extra_sigma=F.ORGAN_IMAGE_SD), None, 0) for r in null_s]))


def organ_table(results):
    rows = pd.DataFrame([x for r in results for x in r["organs"]])
    rows["error"] = np.log(rows.estimate / rows.truth)
    rows["z"] = rows.error / rows.sd_log
    return rows.groupby("structure").agg(bias=("error", "median"), scatter=("error", spread_of), coverage=("z", lambda z: (z.abs() < 1.96).mean()), z_sd=("z", "std")).loc[ORGANS]


def spread_of(x):
    return 1.4826 * np.median(np.abs(x - np.median(x)))


display(pd.concat({"Student-t on the spleen": organ_table(sim0), "Gaussian, nb17": organ_table(gauss)}).round(3))

## 6. Lesions: low tail and viable fraction, against the RC trend

nb17 found a trend of the real leave-one-out residual with recovery; the first table shows it against volume. Models, each added to the censored likelihood; the gain is in $-\log L$, on the training split and on the simulated cohort, where neither necrosis nor mask bias exists, so it measures false positives:

- **Low tail:** a fraction $\pi$ of lesions from a low component of $[R]$, at least a factor $e$ below the median. Against it: the same data can be explained by a wider flow population, which is refitted too.
- **Viable fraction:** $\log y=\log C+\log v$, $\log v=-a\,\max(0,\log(V/V_0))$, $V_0$ 10 and 30 mL, no source. A necrotic core lowers the mean over a large mask.
- **Mask bias (competing explanation):** $\log y=\log C+b\log(1/RC)$. The mask of a small lesion comes from a threshold on the PET and selects hot voxels, so $y/RC$ of a uniform sphere over-corrects. A hypothesis of nb17.
- **Free by recovery class:** three offsets, the shape without a hypothesis.

Each population is refitted, no hyperprior, to compare likelihoods only.

In [ ]:
LESION_STARTS = [(0.4, 0.5), (0.8, 0.6), (1.4, 0.7)]


def lesion_eb_job(job):
    tensors, start, hyperprior, low_tail = job
    return F.eb_lesions_bounded(tensors, start, hyperprior, low_tail)


def fit_lesion_population(results, hyper, hyperprior=True, low_tail=True, offset=None):
    tensors = F.lesion_offset_tensors(results, offset) if offset else [r["likelihood"] for r in results if r["likelihood"].shape[1]]
    starts = [replace(hyper, tau_between=a, tau_within=b, pi_low=max(hyper.pi_low, 0.05) if low_tail else 0.0) for a, b in LESION_STARTS]
    with mp.get_context("fork").Pool(3) as workers:
        solutions = workers.map(lesion_eb_job, [(tensors, s, hyperprior, low_tail) for s in starts])
    best = min(solutions, key=lambda s: s["nll"])
    return best["hyper"], best["nll"], max(s["nll"] for s in solutions) - best["nll"]


real_lesions = pd.DataFrame([x for r in real0 for x in r["lesions"]])
real_lesions["z"] = stats.norm.ppf(real_lesions.pit.clip(1e-6, 1 - 1e-6))
by_volume = real_lesions.groupby(pd.cut(real_lesions.volume_ml, [0, 0.5, 1, 2, 5, 10, 30, 100, 1e5]), observed=True).agg(n=("z", "size"), rc=("rc", "median"), z_mean=("z", "mean"), z_sd=("z", "std"), below_5=("pit", lambda p: (p < 0.05).mean()), above_95=("pit", lambda p: (p > 0.95).mean()), upper_only=("lesion_class", lambda c: (c == "upper bound only").mean()))
by_volume.to_csv(OUT / f"{TAG}_real_lesions_by_volume.csv")
display(by_volume.round(3))
print("Classes, real, full topology:", real_lesions.lesion_class.value_counts(normalize=True).round(3).to_dict())


def offset_job(job):
    name, kind, reference = job
    results = {"real": real0, "null": sim0}[name]
    base = base_population[name]
    out = F.lesion_offset_test(results, kind, base, reference)
    return dict(cohort=name, **out)


def population_job(job):
    name, low_tail = job
    results = {"real": real0, "null": sim0}[name]
    tensors = [r["likelihood"] for r in results if r["likelihood"].shape[1]]
    return name, low_tail, min((F.eb_lesions_bounded(tensors, F.LesionHyper(float(np.log(80)), a, b), False, low_tail) for a, b in LESION_STARTS[:2]), key=lambda s: s["nll"])


def models():
    with pool() as workers:
        populations = workers.map(population_job, [(n, low) for n in ("real", "null") for low in (False, True)], chunksize=1)
    return populations


populations = cached("lesion_models_population", models)
base_population = {name: sol["hyper"] for name, low, sol in populations if not low}
rows = [dict(cohort=name, model="low tail" if low else "no term", nll=sol["nll"], median=np.exp(sol["hyper"].mu), tau_between=sol["hyper"].tau_between, tau_within=sol["hyper"].tau_within, flow=np.exp(sol["hyper"].mu_flow), tau_flow=sol["hyper"].tau_flow, pi_low=sol["hyper"].pi_low, low_median=np.exp(sol["hyper"].mu_low) if low else np.nan) for name, low, sol in populations]


def offsets_build():
    with pool() as workers:
        return workers.map(offset_job, [(n, k, v) for n in ("real", "null") for k, v in (("viable", 10.0), ("viable", 30.0), ("mask", 10.0), ("bins", 10.0))], chunksize=1)


offset_results = cached("lesion_models_offsets", offsets_build)
rows += [dict(cohort=x["cohort"], model=f"{x['kind']}" + (f" V0={x['reference_ml']:.0f}" if x["kind"] == "viable" else ""), nll=x["nll"], value=np.round(x["value"], 3).tolist(), median=np.exp(x["hyper"].mu), tau_between=x["hyper"].tau_between, tau_within=x["hyper"].tau_within, flow=np.exp(x["hyper"].mu_flow), tau_flow=x["hyper"].tau_flow) for x in offset_results]
def combined_job(job):
    name, value = job
    results = {"real": real0, "null": sim0}[name]
    tensors = F.lesion_offset_tensors(results, F.LesionOffset("mask", (value,)))
    return name, value, min((F.eb_lesions_bounded(tensors, F.LesionHyper(float(np.log(80)), a, b), False, True) for a, b in LESION_STARTS[:2]), key=lambda s: s["nll"])


mask_values = {x["cohort"]: x["value"][0] for x in offset_results if x["kind"] == "mask"}
with pool() as workers:
    combined = cached("lesion_models_combined", lambda: workers.map(combined_job, list(mask_values.items()), chunksize=1))
rows += [dict(cohort=name, model="mask + low tail", nll=sol["nll"], value=[round(value, 3)], median=np.exp(sol["hyper"].mu), tau_between=sol["hyper"].tau_between, tau_within=sol["hyper"].tau_within, flow=np.exp(sol["hyper"].mu_flow), tau_flow=sol["hyper"].tau_flow, pi_low=sol["hyper"].pi_low, low_median=np.exp(sol["hyper"].mu_low)) for name, value, sol in combined]
model_table = pd.DataFrame(rows)
model_table["gain"] = model_table.groupby("cohort").nll.transform(lambda v: v.iloc[0] - v)
model_table.to_csv(OUT / f"{TAG}_lesion_models.csv", index=False)
display(model_table.round(3))
edge = pd.DataFrame([x for rows_ in cached("lesion_edge", lambda: run("stage_lesion_edge", [(s, TEST) for s in TRAIN])) for x in rows_])
edge["log_ratio"] = np.log(edge.observed / edge.expected)
edge_table = edge.groupby(pd.cut(edge.volume_ml, [0, 0.5, 1, 2, 5, 10, 30, 100, 1e5]), observed=True).agg(n=("rc", "size"), rc=("rc", "median"), observed=("observed", "median"), expected=("expected", "median"), log_ratio=("log_ratio", "median"))
edge_table["z_mean_of_the_table_above"] = by_volume.z_mean.to_numpy()
edge_table.to_csv(OUT / f"{TAG}_lesion_edge.csv")
display(edge_table.round(3))
print("Smallest lesions by scanner:", edge[edge.volume_ml <= 0.5].groupby("scanner").log_ratio.median().round(3).to_dict(), "| above 10 mL:", edge[edge.volume_ml > 10].groupby("scanner").log_ratio.median().round(3).to_dict())
if os.environ.get("NB18_STOP"):
    raise RuntimeError("stop after the lesion models")

**Independent check of the mask hypothesis, without any fit** (last table). Activity in the first voxel layer outside the mask over the mask, background removed, against a uniform lesion of exactly that mask blurred by the PSF. Above 10 mL they agree, so the assumed FWHM holds at the edges. The smallest masks have more activity outside than a lesion of that extent can give, while the masks of 0.5-5 mL have less. The sign follows the leave-one-out residual volume by volume.

**Decision taken before the empirical Bayes:** the mask term enters, with $b$ estimated once by each cohort at its first pass, so the simulated cohorts test it where the true value is 0. **The viable fraction does not enter**, as the likelihood gives it nothing. The low tail stays in the population, constrained at least a factor $e$ below the median so that it cannot turn into a second main component.

## 7. Empirical Bayes validated on simulated cohorts, full topology

The procedure of the real run: organs by subject (kidneys, liver, spleen, $f_{SAL}$) with $[R_{SAL}]$ frozen, scanner scale and blood offset accumulated over passes, lesions with the low tail and the mask term. Therapy **from the posterior**, before and after. **Null:** the priors. **Shifted:** kidneys × 1.35, liver × 1.1, spleen × 2.4, $f_{SAL}$ × 0.5, lesions median 60, flow 0.30, and 10% non-avid lesions around 3 nmol/L.

In [ ]:
def blood_step(rows, blood_offsets, scale_step):
    return {s: blood_offsets.get(s, 0.0) + float(rows[rows.scanner == s].blood_vs_population.median()) - scale_step[s] for s in rows.scanner.unique()}


def run_eb(jobs_for, name, passes=2):
    def build():
        priors, offsets, blood, hyper = {}, {}, {}, F.LesionHyper()
        adopted = None
        history, fits = [], []
        scanners = None
        for step in range(passes + 1):
            final = step in (0, passes)
            settings = dict(priors=priors or None, offsets=offsets, blood_offsets=blood)
            if adopted:
                settings["lesion_offset"] = adopted
            results = jobs_for(settings, hyper, (30 if not LIMIT else 6) if final else 0)
            fits.append(dict(results=[{k: v for k, v in r.items() if k not in ("likelihood", "organ_grid", "table", "lesions")} for r in results] if final else None))
            if step == passes:
                fits[-1]["hyper"] = hyper
                fits[-1]["likelihood_results"] = results
                break
            scanners = sorted({r["scanner"] for r in results})
            organ = F.eb_organs(results, priors or None, scanners, keys=KEYS, by_subject=True)
            priors = organ["priors"]
            offsets = {s: offsets.get(s, 0.0) + organ["scale"][s] for s in scanners}
            rows = pd.DataFrame([r["row"] for r in results])
            blood = blood_step(rows, blood, organ["scale"])
            first_offset = None
            if adopted is None:
                base, *_ = fit_lesion_population(results, hyper, hyperprior=False, low_tail=False)
                test = F.lesion_offset_test(results, "mask", base)
                adopted = first_offset = F.LesionOffset("mask", test["value"])
            hyper, nll, disagreement = fit_lesion_population(results, hyper, offset=first_offset)
            history.append(dict(priors=priors, offsets=dict(offsets), blood=dict(blood), hyper=hyper, nll=nll, disagreement=disagreement, pull=organ["pull"], mask_b=adopted.value[0]))
            print(name, "pass", step + 1, {k: (round(v[0], 3), round(v[1], 3)) for k, v in priors.items()}, {k: round(v, 3) for k, v in offsets.items()}, {k: round(v, 3) for k, v in blood.items()}, f"mask b {adopted.value[0]:.3f} lesions {np.exp(hyper.mu):.1f} tb {hyper.tau_between:.2f} tw {hyper.tau_within:.2f} flow {np.exp(hyper.mu_flow):.3f} tf {hyper.tau_flow:.2f} pi {hyper.pi_low:.3f} low {np.exp(hyper.mu_low):.1f} starts {disagreement:.2f}")
        return dict(history=history, fits=fits)
    return cached(f"eb_{name}", build)


def simulated_jobs(cohort):
    return lambda settings, hyper, predict: run("stage_fit_full", [(r, settings, hyper, predict) for r in cohort], chunksize=1)


eb_null = run_eb(simulated_jobs(null_s), "null")
eb_shift = run_eb(simulated_jobs(shifted_s), "shifted")
rows = []
for name, eb, truth in [("null", eb_null, F.PRIORS), ("shifted", eb_shift, {**F.PRIORS, **SHIFTED_ORGANS})]:
    last = eb["history"][-1]
    for k in KEYS:
        rows.append(dict(cohort=name, parameter=k, start=F.PRIORS[k][0], truth=truth[k][0], fitted=last["priors"][k][0], ratio=last["priors"][k][0] / truth[k][0], start_sd=F.PRIORS[k][1], truth_sd=truth[k][1], fitted_sd=last["priors"][k][1]))
population = pd.DataFrame(rows)
population.to_csv(OUT / f"{TAG}_eb_simulated_population.csv", index=False)
display(population.round(3))
display(pd.DataFrame({name: eb["history"][-1]["offsets"] for name, eb in [("null scale", eb_null), ("shifted scale", eb_shift)]} | {name: eb["history"][-1]["blood"] for name, eb in [("null blood", eb_null), ("shifted blood", eb_shift)]}).assign(truth_scale_centred=lambda d: pd.Series(F.SCANNER_EFFECT) - np.mean([F.SCANNER_EFFECT[r["geometry"].scanner] for r in null_s])).round(3))


def therapy_summary(results):
    t = pd.DataFrame([x for r in results for x in r["therapy"]])
    out = {}
    for structure in ORGANS + ["lesions"]:
        pet = t[(t.structure == structure) & (t.tag == "PET posterior")]
        none = t[(t.structure == structure) & (t.tag == "no PET")]
        out[structure] = dict(median_pet=pet.ratio.median(), mean_pet=pet.mean_ratio.median(), spread_pet=spread(pet.ratio), cover_68=pet.inside_68.mean(), cover_95=pet.inside_95.mean(), median_no_pet=none.ratio.median(), spread_no_pet=spread(none.ratio))
    return pd.DataFrame(out).T


therapy = pd.concat({f"{name}, {tag}": therapy_summary(eb["fits"][index]["results"]) for name, eb in [("null", eb_null), ("shifted", eb_shift)] for tag, index in [("before", 0), ("after", -1)]})
therapy.to_csv(OUT / f"{TAG}_eb_simulated_therapy.csv")
display(therapy.round(3))
errors = pd.concat({f"{name}, {tag}": pd.DataFrame([r["row"] | dict(gfr_error=r["gfr_error"], hematocrit_error=r["hematocrit_error"]) for r in eb["fits"][index]["results"]])[["gfr_error", "hematocrit_error"]].median() for name, eb in [("null", eb_null), ("shifted", eb_shift)] for tag, index in [("before", 0), ("after", -1)]}, axis=1)
print("Posterior mode against truth, log")
display(errors.round(3))
display(pd.concat({f"{name}, {tag}": organ_table(eb["fits"][index]["results"]) for name, eb in [("null", eb_null), ("shifted", eb_shift)] for tag, index in [("before", 0), ("after", -1)]}).round(3))

## 8. Empirical Bayes on the real training split

**A use of the training data; test split not read.** Same procedure as §7: two passes for organs, scanner scale, blood offset and the lesion population; final pass with therapy from the posterior. $[R_{SAL}]$ frozen.

In [ ]:
def real_jobs(settings, hyper, predict):
    results = run("stage_diagnostic_real", [(s, TEST, {**{k: v for k, v in settings.items()}, "hyper": hyper, "predict": predict}) for s in TRAIN], chunksize=1)
    assert not TEST & {r["study_id"] for r in results}
    return results


eb_real = run_eb(real_jobs, "real")
final_results = eb_real["fits"][-1]["likelihood_results"]
final_rows = pd.DataFrame([r["row"] for r in final_results])
final_rows.to_csv(OUT / f"{TAG}_real_after_eb.csv", index=False)
last = eb_real["history"][-1]
population_real = pd.DataFrame([dict(parameter=k, start=F.PRIORS[k][0], start_sd=F.PRIORS[k][1], **{f"pass_{i + 1}": h["priors"][k][0] for i, h in enumerate(eb_real["history"])}, fitted_sd=last["priors"][k][1], nb17=dict(kidneys=49.185, liver=1.375, spleen=2.117, f_sal=0.065)[k]) for k in KEYS]).set_index("parameter")
population_real.to_csv(OUT / f"{TAG}_eb_real_population.csv")
display(population_real.round(3))
display(pd.DataFrame(dict(scale=last["offsets"], blood=last["blood"])).round(3))
h = last["hyper"]
lesion_population = pd.DataFrame([dict(cohort=n, mask_b=e["history"][-1]["mask_b"], median=np.exp(e["history"][-1]["hyper"].mu), tau_between=e["history"][-1]["hyper"].tau_between, tau_within=e["history"][-1]["hyper"].tau_within, flow=np.exp(e["history"][-1]["hyper"].mu_flow), tau_flow=e["history"][-1]["hyper"].tau_flow, pi_low=e["history"][-1]["hyper"].pi_low, low_median=np.exp(e["history"][-1]["hyper"].mu_low), disagreement=e["history"][-1]["disagreement"]) for n, e in [("real", eb_real), ("null, truth 136, 1.19, 0.72, 0.20, 1.12", eb_null), ("shifted, truth 60, 0.8, 0.6, 0.30, 0.9, pi 0.10, low 3", eb_shift)]])
lesion_population.to_csv(OUT / f"{TAG}_eb_lesion_population.csv", index=False)
display(lesion_population.round(3))
keys = ["kidneys", "liver", "spleen", "salivary", "blood"]
display(pd.concat({"before": pd.DataFrame({k: dict(median=real_rows[f"z_{k}"].median(), sd=real_rows[f"z_{k}"].std()) for k in keys}), "after": pd.DataFrame({k: dict(median=final_rows[f"z_{k}"].median(), sd=final_rows[f"z_{k}"].std()) for k in keys})}).round(2))
display(pd.DataFrame({name: {k: d[k].median() for k in ["r_kidneys", "r_liver", "r_spleen", "r_salivary", "gfr", "f_sal", "hematocrit"]} for name, d in [("before", real_rows), ("after", final_rows)]}).round(3))
display(final_rows.groupby("scanner")[["gfr", "hematocrit", "z_blood", "blood_vs_population"]].median().round(3))
final_lesions = pd.DataFrame([x for r in final_results for x in r["lesions"]])
final_lesions["z"] = stats.norm.ppf(final_lesions.pit.clip(1e-6, 1 - 1e-6))
print("Lesions after: PIT below 5%", round((final_lesions.pit < 0.05).mean(), 3), "above 95%", round((final_lesions.pit > 0.95).mean(), 3), "z SD", round(final_lesions.z.std(), 3), "| common error mean", round(final_rows.common_error.mean(), 3), "SD", round(final_rows.common_error.std(), 3))
display(final_lesions.groupby(pd.cut(final_lesions.volume_ml, [0, 0.5, 1, 2, 5, 10, 30, 100, 1e5]), observed=True).agg(n=("z", "size"), z_mean=("z", "mean"), below_5=("pit", lambda p: (p < 0.05).mean()), above_95=("pit", lambda p: (p > 0.95).mean())).round(3))
print("Classes after:", final_lesions.lesion_class.value_counts(normalize=True).round(3).to_dict())
tiac = pd.DataFrame([{**{k: r["therapy"][k] for k in ORGANS}, "lesion_median": float(np.median(r["therapy"]["lesions"])) if len(r["therapy"]["lesions"]) else np.nan, "ess": r["therapy"]["ess"]} for r in final_results])
tiac.to_csv(OUT / f"{TAG}_real_tiac_posterior.csv", index=False)
print("Real posterior TIAC, h: median and spread between patients")
display(pd.DataFrame(dict(median=tiac[ORGANS + ["lesion_median"]].median(), spread=[spread(tiac[c].dropna()) for c in ORGANS + ["lesion_median"]])).round(3))
print("Importance ESS of 200: median", round(tiac.ess.median(), 1), "| p10", round(tiac.ess.quantile(0.1), 1))
fig, axes = plt.subplots(1, 2, figsize=(9, 2.9), constrained_layout=True)
axes[0].hist(real_lesions.pit, bins=20, density=True, color=COLORS[1], alpha=0.75, label="Start priors")
axes[0].hist(final_lesions.pit, bins=20, density=True, color=COLORS[0], alpha=0.75, label="After empirical Bayes")
axes[0].axhline(1, color=MUTED, lw=0.8)
axes[0].set(xlabel="Leave-one-out PIT of the lesion datum", ylabel="Density")
axes[0].legend(frameon=False)
axes[0].set_title("Real lesions", loc="left")
axes[1].hist(real_rows.common_error.dropna(), bins=20, color=COLORS[1], alpha=0.75, label="Start priors")
axes[1].hist(final_rows.common_error.dropna(), bins=20, color=COLORS[0], alpha=0.75, label="After empirical Bayes")
axes[1].set(xlabel=r"Posterior mean of the common error $\delta$", ylabel="Patients")
axes[1].legend(frameon=False)
axes[1].set_title("Common error", loc="left")
fig.savefig(OUT / f"{TAG}_real_lesions_after.png", dpi=160)

## 9. The lesion median: profile on the final likelihoods

What the data allow for the median, same corrected profile as nb17 §5, no hyperprior, low tail at its fitted value, on the real training split and on the simulated cohorts with the same procedure.

In [ ]:
profile_sets = {"real": eb_real, "null": eb_null, "shifted": eb_shift}


def profile_job18(job):
    name, median = job
    results = profile_sets[name]["fits"][-1]["likelihood_results"]
    tensors = [r["likelihood"] for r in results if r["likelihood"].shape[1]]
    start = profile_sets[name]["history"][-1]["hyper"]
    solution = F.eb_lesions_bounded(tensors, start, False, start.pi_low > 0, fixed_mu=float(np.log(median)))
    return dict(cohort=name, median=median, nll=solution["nll"], tau_between=solution["hyper"].tau_between, tau_within=solution["hyper"].tau_within, flow=np.exp(solution["hyper"].mu_flow), pi_low=solution["hyper"].pi_low)


def profiles18():
    grid = np.geomspace(15, 600, 13)
    with pool() as workers:
        return workers.map(profile_job18, [(name, m) for name in profile_sets for m in grid], chunksize=1)


profile = pd.DataFrame(cached("lesion_profile", profiles18))
profile["delta_nll"] = profile.nll - profile.groupby("cohort").nll.transform("min")
profile.to_csv(OUT / f"{TAG}_lesion_profile.csv", index=False)
summary = []
for name, sub in profile.groupby("cohort", sort=False):
    x, y = np.log(sub["median"].to_numpy()), sub.delta_nll.to_numpy()
    k = int(np.argmin(y))
    lo = next((np.exp(np.interp(1.92, [y[i], y[i - 1]], [x[i], x[i - 1]])) for i in range(k, 0, -1) if y[i - 1] > 1.92), np.exp(x[0]))
    hi = next((np.exp(np.interp(1.92, [y[i], y[i + 1]], [x[i], x[i + 1]])) for i in range(k, len(x) - 1) if y[i + 1] > 1.92), np.exp(x[-1]))
    summary.append(dict(cohort=name, best=np.exp(x[k]), low=lo, high=hi, tau_between=sub.tau_between.iloc[k], flow=sub.flow.iloc[k], pi_low=sub.pi_low.iloc[k]))
display(pd.DataFrame(summary).round(3))
fig, ax = plt.subplots(figsize=(5, 2.9), constrained_layout=True)
for (name, sub), color in zip(profile.groupby("cohort", sort=False), COLORS):
    ax.plot(sub["median"], sub.delta_nll, color=color, lw=1.6, label=name)
ax.axhline(1.92, color=MUTED, lw=0.8, ls=":")
ax.set_xscale("log")
ticks = [20, 50, 100, 200, 500]
ax.set_xticks(ticks, [str(t) for t in ticks])
ax.minorticks_off()
ax.set(ylim=(0, 25), xlabel="Population median of lesion [R] (nmol/L)", ylabel=r"Profile $-\log L$")
ax.legend(frameon=False)
ax.set_title("How much the data fix the lesion median", loc="left")
fig.savefig(OUT / f"{TAG}_lesion_profile.png", dpi=160)

## 9b. What fixes the lesion median on real data?

At 2 nmol the 1 h concentration of most lesions follows the flow; density only caps it. So a low median on real data can have one of two meanings: density truly low, or the **upper tail of the real lesion data is lighter than a log-normal flow gives**, and the fit explains the missing tail by saturation. **Test:** the same likelihoods with the flow population capped at 0.2, 0.5, 1 mL/g/min (caps fixed here, not fitted; the literature range used for the prior is 0.02-1.6) and the median free or fixed at the start value 136. If the data held information on density, the median should not depend on the shape of the flow population.

**And what it does at therapy.** Lesion over kidney TIAC per mL at 100 nmol, on 30 real geometries, kidney at the fitted median, at population values. The published ratio of absorbed doses for [177Lu]Lu-PSMA-617 (Ells et al. 2024, Table 3) is 3.57-4.19 over 0.58 Gy/GBq for lesions over kidneys. Different ligand, and doses include S-value differences, so an order of magnitude only.

In [ ]:
CAPS = [0.2, 0.5, 1.0, np.inf]
cap_sets = {"real": eb_real, "null": eb_null, "shifted": eb_shift}


def cap_job(job):
    name, cap, mu = job
    results = cap_sets[name]["fits"][-1]["likelihood_results"]
    tensors = [r["likelihood"] for r in results if r["likelihood"].shape[1]]
    start = replace(cap_sets[name]["history"][-1]["hyper"], pi_low=0.0, flow_cap=cap)
    best = min((F.eb_lesions_bounded(tensors, replace(start, tau_between=a, tau_within=b), False, False, fixed_mu=None if mu is None else float(np.log(mu)), flow_cap=cap) for a, b in LESION_STARTS[:2]), key=lambda x: x["nll"])
    return dict(cohort=name, cap=cap, fixed=mu, nll=best["nll"], median=np.exp(best["hyper"].mu), tau_between=best["hyper"].tau_between, tau_within=best["hyper"].tau_within, flow=np.exp(best["hyper"].mu_flow), tau_flow=best["hyper"].tau_flow)


def caps_build():
    with pool() as workers:
        return workers.map(cap_job, [(n, c, m) for n in cap_sets for c in CAPS for m in (None, F.TUMOUR_MEDIAN)], chunksize=1)


caps = pd.DataFrame(cached("lesion_caps", caps_build))
caps["cost_of_fixing_at_136"] = caps.groupby(["cohort", "cap"]).nll.transform(lambda v: v.iloc[1] - v.iloc[0])
caps.to_csv(OUT / f"{TAG}_lesion_caps.csv", index=False)
display(caps[caps.fixed.isna()].drop(columns="fixed").round(3))


def ratio_job(job):
    study, r, f = job
    geometry = F.load_geometry(study)
    if not len(geometry.lesions_ml):
        return None
    n = len(geometry.lesions_ml)
    prior = eb_real["history"][-1]["priors"]
    out = F.therapy_tiac(geometry, dict(kidneys=prior["kidneys"][0], liver=prior["liver"][0], spleen=prior["spleen"][0], salivary=F.PRIORS["salivary"][0]), np.full(n, r), f_sal=prior["f_sal"][0], gfr=F.PRIORS["gfr"][0], hematocrit=F.PRIORS["hematocrit"][0], f_lesions=np.full(n, f), l_lesions=np.full(n, F.PRIORS["l_tumor"][0]), l_kidney=F.PRIORS["l_kidney"][0], l_salivary=F.PRIORS["l_salivary"][0], topology="full")
    return dict(study=study, r=r, f=f, ratio=float(np.median(out["lesions"] / geometry.lesions_ml) / (out["kidneys"] / geometry.kidneys_ml)))


def ratios_build():
    hyper = eb_real["history"][-1]["hyper"]
    with pool() as workers:
        out = workers.map(ratio_job, [(s, r, f) for s in TRAIN[::6] for r, f in [(float(np.exp(hyper.mu)), float(np.exp(hyper.mu_flow))), (F.TUMOUR_MEDIAN, float(np.exp(hyper.mu_flow))), (F.TUMOUR_MEDIAN, 0.2), (F.TUMOUR_MEDIAN * 5, 0.2)]], chunksize=2)
    return [x for x in out if x]


ratios = pd.DataFrame(cached("lesion_kidney_ratio", ratios_build))
ratio_table = ratios.groupby(["r", "f"]).ratio.median().rename("lesion over kidney TIAC per mL").reset_index()
ratio_table.to_csv(OUT / f"{TAG}_lesion_kidney_ratio.csv", index=False)
display(ratio_table.round(3))
real_ratio = pd.DataFrame([dict(ratio=np.median(r["therapy"]["lesions"] / r["volume"]) / (r["therapy"]["kidneys"] / r["row"]["kidneys_ml"])) for r in final_results if len(r["volume"])])
print("Real posterior, patient median of lesion over kidney TIAC per mL:", round(real_ratio.ratio.median(), 3), "| q10, q90", real_ratio.ratio.quantile([0.1, 0.9]).round(3).tolist())
slopes = []
for r in final_results:
    g = r["organ_grid"]["kidneys"]
    i = int(np.argmin(np.abs(g["model"] - g["data"])))
    i = min(max(i, 1), len(F.ORGAN_GRID_LOG) - 2)
    slopes.append((g["model"][i + 1] - g["model"][i - 1]) / (F.ORGAN_GRID_LOG[i + 1] - F.ORGAN_GRID_LOG[i - 1]))
print("Kidney datum sensitivity to density at the fit, d log datum / d log [R_K]: median", round(np.median(slopes), 3), "q10, q90", np.round(np.quantile(slopes, [0.1, 0.9]), 3).tolist())

## 9c. Kidney shell excess: sensitivity analysis only

The proposal of §3, not part of the main arm. From the kidney datum, subtract the activity above parenchyma in the shell, beyond the median share a urine-free simulated PET gives. Same real PETs, start priors, first pass of the organ empirical Bayes in both arms.

In [ ]:
floor = float(SIMK.shell_excess_fraction.median())
subtracted = cached("kidney_excess_sensitivity", lambda: run("stage_diagnostic_real", [(s, TEST, dict(kidney_excess_floor=floor)) for s in TRAIN]))
scanners_real = sorted(real_rows.scanner.unique())
base_eb = F.eb_organs(real0, None, scanners_real, keys=KEYS, by_subject=True)["priors"]
sens_eb = F.eb_organs(subtracted, None, scanners_real, keys=KEYS, by_subject=True)["priors"]
display(pd.DataFrame({"main, first pass": {k: base_eb[k][0] for k in KEYS}, "main SD": {k: base_eb[k][1] for k in KEYS}, "shell excess subtracted": {k: sens_eb[k][0] for k in KEYS}, "subtracted SD": {k: sens_eb[k][1] for k in KEYS}}).round(3))
compare = real_rows[["study_id", "r_kidneys", "datum_kidneys"]].merge(pd.DataFrame([r["row"] for r in subtracted])[["study_id", "r_kidneys", "datum_kidneys"]], on="study_id", suffixes=("", "_sub"))
compare["d_datum"] = np.log(compare.datum_kidneys_sub / compare.datum_kidneys)
compare["d_r"] = np.log(compare.r_kidneys_sub / compare.r_kidneys)
compare.to_csv(OUT / f"{TAG}_kidney_excess_sensitivity.csv", index=False)
display(compare[["d_datum", "d_r"]].quantile([0.1, 0.5, 0.9, 1.0]).round(4))
top = compare[compare.d_datum <= compare.d_datum.quantile(0.1)]
print("Lowest decile of the datum change:", round(top.d_datum.median(), 3), "in the datum,", round(top.d_r.median(), 3), "in the fitted density")

## 10. Results, and closing of the real-PET fit

All stages 180 of 180, on simulated and real data, including the two patients that failed.

### What this round found

**Failing patient (§1).** The optimiser reached a hematocrit of about 0.9, not an anatomical problem. Bounded to 0.20-0.65 in fit and sampler; both patients now fit, at 0.41.

**Blood: measurement, not model (§2).** Datum over the model at the population nuisance, scanner scale removed, per-scanner PSF: Biograph20_mCT -0.19, Biograph64 +0.06, Discovery690 -0.28 in log (Kruskal-Wallis p = 5e-20; simulation -0.00, +0.02, -0.00, p = 0.83). The PSF changes recovery by 2% and removes nothing. Uptake time, TLV and weight do not absorb it (uptake coefficient 0.03). By the rule set beforehand, **no early-clearance nuisance was added.** Instead a blood offset per scanner: GFR by scanner goes from 115, 93, 122 to 98, 88, 93; hematocrit 0.42-0.43. **The physical cause is not identified:** each scanner has one voxel grid and its own patients, so scanner, reconstruction and grid cannot be separated. The offsets also remove any uniform component, so **the blood pool now says nothing on the population level of GFR.**

**Kidneys: the collecting system is not the explanation (§3).** Label 2 is close to empty (median 0 mL), so the pelvis evidently lies outside the CT mask, in the shell. Renal sinus over parenchyma 0.41 on real data against 0.46 on simulated, which has no urine; sinus share of the datum 3.7% against 4.7%. Activity above parenchyma in the whole shell: median 0.6%, 90th percentile 4.1% (simulated 0.1%, 1.8%). Correlation with fitted $[R_K]$: at most |0.12| for the excess, not significant. The -0.26 of sinus over parenchyma has the wrong sign for urine: hot parenchyma is its denominator. **The real cause of the unstable $[R_K]$ is sensitivity:** $d\log\text{datum}/d\log R_K$ is 0.18 at the fit (q10 0.014, q90 0.58), so a 1% error of the datum is about 6% in density before the prior (with the prior, about 1.5 in §9c). *Fix, as a sensitivity analysis only (§9c), not in the main arm:* subtract the per-patient shell excess above the simulated level from the datum. It moves the kidney population median from 51.1 to 48.3 (-5.5%, SD 1.20 to 1.18) at the first pass and no other organ; the datum changes by -0.6% in median and -8.7% in the most affected decile, the fitted density by -14% there.

**Full topology and posterior (§4, §7).** Importance sampling agrees with Metropolis (median shift below 0.08 SD, SD ratio 0.98-1.07, ESS median 81 of 200, minimum 10), so it is the sampler for the cohort. Posterior median over MAP in TIAC: kidneys 1.072 ± 0.014, liver 1.032 ± 0.014, spleen 1.003 ± 0.015, glands 0.986 ± 0.016. **The excess of nb13-nb16 is closed:**

| TIAC median against truth, from the posterior | Kidneys | Liver | Spleen | Glands | Lesions |
|---|---|---|---|---|---|
| Null, start priors | 1.07 | 1.02 | 1.07 | 0.99 | 1.00 |
| Null, after EB | 0.94 | 1.01 | 1.06 | 0.94 | 0.92 |
| Shifted, start priors | 0.87 | 0.97 | 0.91 | 0.89 | 1.24 |
| Shifted, after EB | 1.02 | 1.04 | 1.00 | 0.97 | 0.74 |
| Spread after EB, null / shifted | 0.43 / 0.39 | 0.25 / 0.22 | 0.24 / 0.26 | 0.49 / 0.40 | 0.60 / 0.74 |
| Spread no PET, null / shifted | 0.46 / 0.39 | 0.88 / 0.53 | 0.83 / 0.70 | 0.47 / 0.43 | 1.26 / 1.29 |

Intervals: with 30 draws the attainable coverage is about 0.64 and 0.89 for nominal 68% and 95%. Observed after EB: kidneys 0.63-0.67, 0.88-0.91; liver 0.62-0.66, 0.89-0.92; spleen 0.55-0.58, 0.86; glands 0.55-0.63, 0.86-0.90; lesions 0.62, 0.90 in the null and **0.48, 0.78 in the shifted cohort**. Posterior mean above median by 5-11%: report which one is used.

**Spleen, Student-t (§5).** Under the t, 0.91 of the sampled density intervals contain the truth, and the scatter goes from 0.31 to 0.285. The Laplace coverage (0.81) is not a valid check for a t likelihood. Bias +0.067 stays in the null, 0.033 after EB, -0.05 in the shifted.

**Organ empirical Bayes, recovery (§7), truth to fitted.** Liver and spleen within 1-12%, $f_{SAL}$ 3-10% low; kidneys **11-15% low in both cohorts**; spleen SD 13-16% low. Scanner scale within 0.03 of the centred truth; simulated blood offsets at most 0.05.

**Lesions (§6, §8, §9).**

- *Viable fraction: rejected.* Gain 0.01 and 0 in $-\log L$ at $V_0$ 10 and 30 mL.
- *The RC trend is a mask bias of the smallest lesions:* gain 316 for one parameter, $b=0.40$, against 0 in the simulation, where $b=0$. Free offsets give +0.69 below RC 0.3. An independent check supports it: outside the smallest masks there is more activity than that extent gives (+0.13), while above 10 mL the assumed PSF holds (-0.03, +0.01). That check supports the sign and not the size. A residual trend with volume stays after the term (+0.32 below 0.5 mL, -0.32 at 1-2 mL, +0.34 to +0.41 above 10 mL). $b$ is fitted on the training split.
- *Low tail: not needed on real data.* Gain 2.1, and with the mask term $\pi=0.002$. In simulation: null 0.002 (no false positive), shifted 0.055 at 3.8 nmol/L against the true 0.10 at 3. The upper-only class falls from 10.5% to 0.6%, because the whole population moved, not because of a tail.
- *Validation of the lesion EB: partial.* Null 105 [58, 154] against 136, flow 0.20. **Shifted 34 [26, 40] against 60, flow 0.43 against 0.30: the truth is excluded, and lesion TIAC goes to 0.74.** In simulation the median also depends strongly on the shape of the flow population (capping flow: 96 to 1155 in the null).
- *Real result: median 6.7 nmol/L, $\tau_{between}$ 1.08, $\tau_{within}$ 0.29 (near its bound), flow 0.06, $\tau_{flow}$ 0.81.* The profile is at the edge of its grid (best 15, the lowest value; fixing the median at the start value 136 costs 61-85 in $-\log L$). It is robust to capping flow (5.9-7.8), so not the shape of the flow tail. Leave-one-out PIT calibrated: 5.0% and 4.6% in the tails, $z$ SD 0.995, $\delta$ mean -0.024, SD 0.135. **So the fit describes the 1 h lesion data well.**
- ***But this population is not credible at therapy.*** Lesion over kidney TIAC per mL: 0.13 (q10 0.03, q90 1.2) with the real posterior, 0.22 at its population values, 4.0 at the start values (136, flow 0.2), against about 6-7 from published lesion and kidney doses (Ells 2024, PSMA-617, order of magnitude). **It is low by a factor 30-50.** nb16 showed that the 1 h PET cannot choose the ligand, and the table $C(R,f)$ is built with an assumed one, so a wrong PET-ligand kinetics could read as low density here; this is one candidate among those below, not shown.

**Finding: published I&T parameters applied to a PSMA-11 PET are not consistent with PSMA-617 lesion doses.** Under the Kletting parameters, the 1 h PSMA-11 lesion data give a population that predicts lesion over kidney TIAC per mL low by a factor 30-50 against published PSMA-617 doses. Candidate causes: **ligand** (K_D and internalization of the PET and therapy agents), **perfusion** (flow and its coupling to density in lesions), **PET ligand amount** (the injected nmol of 68Ga-PSMA-11 are assumed, 2 ± 0.5). **Not resolved.**

**Decision, confirmed: the real lesion density population does NOT become a therapy prior.** The starting lesion population stays and is anchored to published lesion doses in plan (b). From the real PET, lesions keep only delivery (flow) and the lower bound on density.

### Real training split: final values

Test split not read; a use of the training data. Studies of a subject as one patient, $[R_{SAL}]$ frozen.

| Parameter | Start (log SD) | Fitted (log SD) | Pass 1 to pass 2 | Status |
|---|---|---|---|---|
| Liver [R] | 1.33 (1.00) | **1.27 (0.34)** | 1.30, 1.27 | usable |
| Spleen [R] | 0.53 (1.20) | **1.93 (0.47)** | 1.98, 1.93 | usable; SD maybe 15% higher |
| $f_{SAL}$ | 0.153 (0.81) | **0.062 (0.43)** | 0.063, 0.062 | usable as a delivery term |
| Kidneys [R] | 25.4 (0.44) | **43.9 (1.11)** | 51.1, 43.9 | population level only, not converged, likely 11-15% low |

### What the real-PET fit is good for

1. **Liver and spleen, per patient.** With PET the TIAC spread falls from 0.53-0.88 to 0.22-0.26. This is where the 1 h PET gives most. Population medians within 12% on simulation and stable across passes.
2. **Population priors of liver, spleen and kidneys, measured** on 132 subjects, instead of 5 patients on another ligand. Median TIAC with the fit is within about 7% of the truth on organs on simulation.
3. **Gland delivery** ($f_{SAL}$), with the plasma curve at the PET. **Not gland TIAC.**
4. **Lesion flow-delivery at 1 h and a lower bound on density.** This is all that lesions keep from the real PET.
5. **A validated machinery:** full topology in fit and generation, posterior sampling checked against Metropolis, error terms measured, scanner in the population model.

### What it is not good for, and the limitations to declare

1. **Washout.** No information on $\lambda_{rel}$ (nb13). All therapy spreads are conditional on that prior, from 5 patients on I&T: **plan (b) sets it.**
2. **Therapy ligand.** $K_D$ and $\lambda_{int}$ are not chosen by the PET (nb16). **All absolute TIAC values here are for the assumed ligand.**
3. **Lesion density.** Lower bound for most lesions. The real population is low by 30-50 at therapy and is not used as a therapy prior. Lesion EB recovers the null but not the shifted cohort.
4. **Glands.** Density at its prior; no gain in TIAC spread (0.49 against 0.47 in the null, 0.40 against 0.43 in the shifted). Gland volume factor 1.7 is an assumption.
5. **Kidneys.** Spread no better than no PET (0.43 against 0.46; 0.39 either way), weak sensitivity, $[R_K]$ not converged between passes, SD 1.1 mixes biology and datum error.
6. **Blood.** Scanner-dependent up to 0.34 in log, cause not identified, handled by offsets. The population GFR comes from the priors, and Table A with the Fresco limitation (nb12). The FWHM per scanner is an assumption, not read from the reconstruction.
7. **Image model in simulation.** Shift of 4 mm SD, Student-t parameters, lesion image errors and common error are measured on our simulation or assumed. Real scanners differ in more than scale and PSF, so scanner recovery is an upper limit.
8. **Mask term on lesions:** one fitted parameter; the residual trend with volume is not closed.
9. **Satellites at therapy above 1 L of TLV:** the fit and these TIACs use satellites, while cohort generation uses coupled lesions there (nb12: up to -14% in TIAC, 12 of 227 studies). Lesion draws use the plasma of the MAP.
10. **Coverage** of organ TIAC a little under nominal on the spleen; lesions under in the shifted case.
11. **Test split never read.** All real values are from the training split; there is no truth on real patients.

**Next: plan (b).** Not started.